# 🧠 Fine-Tuning an LLM for AI Research Paper Summarization (PEFT + LoRA)

---

**Model:** TinyLlama-1.1B &nbsp;|&nbsp; **Dataset:** ArXiv (subset) &nbsp;|&nbsp; **Method:** LoRA / PEFT &nbsp;|&nbsp; **Eval:** ROUGE-1/2/L

---

This notebook demonstrates an end-to-end pipeline for fine-tuning a Large Language Model on AI research abstracts using **Parameter-Efficient Fine-Tuning (PEFT) via LoRA** — covering data preprocessing, prompt formatting, model training, and ROUGE evaluation.

> By the end, you'll see a direct comparison between base model and fine-tuned outputs, with measurable improvements in summary quality, coherence, and relevance.

**Outline:** `01 Load dataset` → `02 Preprocess & format prompts` → `03 Apply LoRA` → `04 Train` → `05 Evaluate` → `06 Compare outputs`

####Environment Setup

In [1]:
!pip install -q transformers datasets peft accelerate evaluate trl rouge_score

In [2]:
import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memory:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
else:
    print("No GPU detected")

GPU: Tesla T4
Memory: 15.64 GB


####Dataset Preparation

In [4]:
from datasets import load_dataset

dataset = load_dataset("ccdv/arxiv-summarization", split="train[:1500]")
val_dataset = load_dataset("ccdv/arxiv-summarization", split="validation[:100]")

In [6]:
print(dataset[0].keys(), "\n")
print("Sample abstract: ", dataset[0]["article"][:300], "\n")
print("Sample summary: ", dataset[0]["abstract"][:300])

dict_keys(['article', 'abstract']) 

Sample abstract:  additive models @xcite provide an important family of models for semiparametric regression or classification . some reasons for the success of additive models are their increased flexibility when compared to linear or generalized linear models and their increased interpretability when compared to fu 

Sample summary:  additive models play an important role in semiparametric statistics . 
 this paper gives learning rates for regularized kernel based methods for additive models . 
 these learning rates compare favourably in particular in high dimensions to recent results on optimal learning rates for purely nonpara


In [8]:
def format_prompt(sample):
  return{
      "text": f"Summarize the following AI research abstract:\n\n{sample['article'][:1024]}\n\nSummary: {sample['abstract']}"
  }

In [9]:
dataset = dataset.map(format_prompt)
val_dataset = val_dataset.map(format_prompt)

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [10]:
from transformers import AutoTokenizer

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]